In [2]:
%pip install nltk
%pip install jieba
%pip install gensim
%pip install Word2Vec


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for Word2Vec: filename=word2vec-0.11.1-py2.py3-none-any.whl size=100915 sha256=35e9acb8d6c1e4a3eef501b6af6fa81d6b43bc9ea4fdfac8015373b71f0e5932
  Stored in directory: /Users/laurenchen/Library/Caches/pip/wheels/30/a1/56/d2cd6e824ce0dc7380f3507658f18d2b073361d14d08e793db
Successfully built Word2Vec

[notice] A new rel

In [2]:
import os
import sqlite3
path = os.path.expanduser('~/Library/Messages/chat.db')
conn = sqlite3.connect(path)
cursor = conn.cursor()
cursor.execute("SELECT text FROM message WHERE text IS NOT NULL")
texts = [row[0] for row in cursor.fetchall()]


In [3]:
import re
import jieba
def clean_text_file(text):
    text = text.strip()
    text = re.sub(r'\s', ' ', text)
    text = text.lower()
    tokens = jieba.lcut(text)
    tokens = [t for t in tokens if t.strip() != '']
    return tokens
tokenized_texts = [clean_text_file(t) for t in texts]
with open('cleaned_tokenized_text.txt', 'w', encoding="utf-8") as f:
    for tokens in tokenized_texts:
        f.write(' '.join(tokens) + '\n')
print("saved")

Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/cm/6sb2v0pj3fj_dsxglfvcw8980000gn/T/jieba.cache
Loading model cost 0.230 seconds.
Prefix dict has been built successfully.


saved


In [4]:
from gensim.models import Word2Vec
sentences = [line.split() for line in open('cleaned_tokenized_text.txt', 'r', encoding="utf-8")]
model = Word2Vec(
    sentences = sentences,
    vector_size = 100,
    window=5,
    min_count = 1,
    workers = 4,
    seed=123
)
model.save("imessage_word2vec.model")


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [5]:
model = Word2Vec.load("imessage_word2vec.model")
similar_words = model.wv.most_similar("night")
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")

morning: 0.7253
week: 0.6932
sunday: 0.6889
day: 0.6779
rehearsal: 0.6768
carvel: 0.6744
concert: 0.6568
round: 0.6484
mall: 0.6457
weke: 0.6337


In [6]:
%pip install pandas scikit-learn plotly


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
vocab_size = len(model.wv.key_to_index)
print(vocab_size)


83028


In [7]:
from gensim.models import Word2Vec
model = Word2Vec.load("imessage_word2vec.model")

In [10]:
words = list(model.wv.key_to_index.keys())[:10000]
vectors = [model.wv[w] for w in words]
with open("vectors.tsv", "w") as f:
    for vec in vectors:
        f.write("\t".join(str(x) for x in vec)+"\n")
with open("metadata.tsv", "w") as f:
    for word in words:
        f.write(word + "\n") 

python -m gensim.scripts.word2vec2tensor -i imessage_word2vec.model -o tsne/
